# 01 · Data collection

Two sources, 2014–2026:

* **Formula 1:** [Jolpica-F1 API](https://github.com/jolpica/jolpica-f1) for race results
  (incl. fastest laps and their average speed) and qualifying (Q1/Q2/Q3 times).
* **MotoGP:** the public results API behind motogp.com, for Q1/Q2 and race classifications
  plus the race's fastest-lap record.

Both clients cache every response and throttle requests. If the companion projects'
caches exist next to this repo, they are reused, so nothing is downloaded twice.

In [1]:
from pathlib import Path

import pandas as pd

from f1_vs_motogp.config import DATA_RAW, ROOT, VENUES
from f1_vs_motogp.data import build_dataset

sibling_f1 = ROOT.parent / "f1-race-outcome-predictor" / "data" / "raw" / "jolpica"
sibling_mgp = ROOT.parent / "motogp-performance-analytics" / "data" / "raw" / "motogp"
f1_cache = sibling_f1 if sibling_f1.exists() else DATA_RAW / "jolpica"
mgp_cache = sibling_mgp if sibling_mgp.exists() else DATA_RAW / "motogp"
tables = build_dataset(2014, 2026, Path(f1_cache), Path(mgp_cache))
{name: df.shape for name, df in tables.items()}

F1 2014 done


F1 2015 done


F1 2016 done


F1 2017 done


F1 2018 done


F1 2019 done


... (17 outputs left out)

MotoGP 2024 done


MotoGP 2025 done


MotoGP 2026 done


{'f1_races': (272, 7),
 'f1_results': (5545, 13),
 'f1_quali': (5524, 6),
 'mgp_races': (242, 10),
 'mgp_results': (5433, 13),
 'mgp_quali': (5954, 7)}

## Races per season

In [2]:
pd.DataFrame({
    "F1 races": tables["f1_races"].groupby("season").size(),
    "MotoGP races": tables["mgp_races"].groupby("season").size(),
})

,F1 races,MotoGP races
season,,
2014,19,18
2015,19,18
2016,21,18
2017,20,18
2018,21,18
2019,21,19
2020,17,14
2021,22,18
2022,22,20


## The shared venues

Eight circuits hosted both championships in this period. Mugello (F1 2020 only), Portimão
(F1 2020–21) and Sepang (F1 to 2017, then again in 2026) give only a handful of pairs.

In [3]:
f1_ids = {v[1]: k for k, v in VENUES.items()}
mgp_names = {v[2]: k for k, v in VENUES.items()}
f1_years = (tables["f1_races"].assign(venue=lambda d: d["circuit_id"].map(f1_ids))
            .dropna(subset=["venue"]).groupby("venue")["season"].apply(sorted))
mgp_years = (tables["mgp_races"].assign(venue=lambda d: d["circuit_name"].map(mgp_names))
             .dropna(subset=["venue"]).groupby("venue")["season"].apply(sorted))
pd.DataFrame({"venue": [VENUES[k][0] for k in VENUES],
              "F1 seasons": [f1_years.get(k, []) for k in VENUES],
              "MotoGP seasons": [mgp_years.get(k, []) for k in VENUES]})

,venue,F1 seasons,MotoGP seasons
0,Circuit of the Americas,"[2014, 2015, 2016, 2017, 2018, 2019, 2021, 202...","[2014, 2015, 2016, 2017, 2018, 2019, 2021, 202..."
1,Silverstone,"[2014, 2015, 2016, 2017, 2018, 2019, 2020, 202...","[2014, 2015, 2016, 2017, 2019, 2021, 2022, 202..."
2,Red Bull Ring,"[2014, 2015, 2016, 2017, 2018, 2019, 2020, 202...","[2016, 2017, 2018, 2019, 2020, 2020, 2021, 202..."
3,Barcelona-Catalunya,"[2014, 2015, 2016, 2017, 2018, 2019, 2020, 202...","[2014, 2015, 2016, 2017, 2018, 2019, 2020, 202..."
4,Lusail (Qatar),"[2021, 2023, 2024, 2025]","[2014, 2015, 2016, 2017, 2018, 2019, 2021, 202..."
5,Sepang,"[2014, 2015, 2016, 2017, 2026]","[2014, 2015, 2016, 2017, 2018, 2019, 2022, 202..."
6,Mugello,[2020],"[2014, 2015, 2016, 2017, 2018, 2019, 2021, 202..."
7,Portimão,"[2020, 2021]","[2020, 2021, 2021, 2022, 2023, 2024, 2025]"
